# Real Data ST Held-Out Prediction: Column Temporal-Cap Ablation

This notebook tests whether Column's small range/nugget comes from using too much temporally unshifted reverse-L conditioning.

Focused comparison:

- `(14, 14, 14)`: baseline Column, same-time/t-1/t-2 all kept.
- `(14,  6,  6)`: keep full same-time structure, but use only the nearest 6 reverse-L candidates from each temporal lag.
- `(14,  0,  0)`: remove temporal conditioning entirely; same-time reverse-L only.

Hybrid baseline is included only as a reference.  Prediction is evaluated on held-out observed points using RMSE and log predictive score.


In [ ]:
import sys, time, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch

LOCAL_SRC = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
AMAREL_SRC = "/home/jl2815/tco"
SRC = AMAREL_SRC if Path(AMAREL_SRC).exists() else LOCAL_SRC
sys.path.insert(0, SRC)

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernels_st_trend_050726 import HybridSTTrendVecchiaFit
from GEMS_TCO.kernels_st_lagcap_050726 import ColumnSTLagCapTrendVecchiaFit

DEVICE = torch.device('cpu')
DTYPE = torch.float64
ROUND_DECIMALS = 4

print('SRC:', SRC)
print('torch:', torch.__version__)
print('device:', DEVICE)


In [ ]:
# Experiment config
YEAR = '2024'
MONTH = 7
DAY_IDX_LIST = [2]  # 0-based: July 3
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]
MM_COND_NUMBER = 100
FIT_SMOOTHS = [0.5]
MEAN_DESIGN = 'hour_spatial'

# Holdout size: stratified by hour.  500/hour gives 4,000 held-out points for one day.
HOLDOUT_PER_HOUR = 500
HOLDOUT_FRAC_PER_HOUR = None
RANDOM_SEED = 20260507

RUN_VARIANTS = [
    'hybrid_base',
    'column_lags_14_14_14',
    'column_lags_14_06_06',
    'column_lags_14_00_00',
]

BASE_COLUMN = {
    'kind': 'column_lagcap',
    'head_right_cols': 0,
    'above_count': 2,
    'right_col_count': 3,
    'include_lag_self': False,
    'target_chunk_size': 512 if DEVICE.type == 'cpu' else 4096,
}

VARIANT_SPECS = {
    'hybrid_base': {
        'kind': 'hybrid',
        'variant': 'hybrid_base',
        'model': 'HybridSTPred_Base_A20_L08F04_C4F03_Op0p063_exactloc',
        'note': 'reference hybrid: same-time A20 + lag self/local/fresh shifted neighbors',
        'nheads': 0, 'limit_A': 20, 'lag1_local_count': 8, 'lag1_fresh_count': 4,
        'lag2_local_count': 4, 'lag2_fresh_count': 3, 'daily_stride': 2,
        'lag1_lon_offset': 0.063, 'spatial_coords_for_shift': None,
    },
}

for name, caps, note in [
    ('column_lags_14_14_14', (14, 14, 14), 'baseline column: same-time/t-1/t-2 all use 14 reverse-L candidates'),
    ('column_lags_14_06_06', (14, 6, 6), 'temporal lags both keep only nearest 6 reverse-L candidates'),
    ('column_lags_14_00_00', (14, 0, 0), 'all temporal conditioning removed; same-time reverse-L only'),
]:
    spec = dict(BASE_COLUMN)
    spec.update({
        'variant': name,
        'model': f"ColumnSTPred_LagCaps_{caps[0]:02d}_{caps[1]:02d}_{caps[2]:02d}_Up2_Right3_head0_realloc",
        'note': note,
        'per_lag_conditioning_counts': caps,
    })
    VARIANT_SPECS[name] = spec

LBFGS_LR = 1.0
LBFGS_STEPS = 5
LBFGS_EVAL = 15
LBFGS_HIST = 10
PRED_CHUNK_SIZE = 1024

INIT_DICT = {
    'sigmasq':    13.059,
    'range_lat':  0.2,
    'range_lon':  0.25,
    'range_time': 1.5,
    'advec_lat':  0.0218,
    'advec_lon': -0.1689,
    'nugget':     0.247,
}
P_LABELS = ['sigmasq', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'nugget']

OUT_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/testing/log')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PREFIX = 'real_st_holdout_lagcap_column_ablation_050726'
OUT_SUMMARY_CSV = OUT_DIR / f'{OUT_PREFIX}_summary.csv'
OUT_PRED_CSV = OUT_DIR / f'{OUT_PREFIX}_predictions.csv'
OUT_PARAM_CSV = OUT_DIR / f'{OUT_PREFIX}_params.csv'

print('variants:', RUN_VARIANTS)
print('holdout per hour:', HOLDOUT_PER_HOUR)
print('summary:', OUT_SUMMARY_CSV)


In [ ]:
def phys_to_log(d):
    phi2 = 1.0 / d['range_lon']
    phi1 = d['sigmasq'] * phi2
    phi3 = (d['range_lon'] / d['range_lat']) ** 2
    phi4 = (d['range_lon'] / d['range_time']) ** 2
    return [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4), d['advec_lat'], d['advec_lon'], np.log(d['nugget'])]


def backmap_params(out_params):
    p = [float(x.item() if isinstance(x, torch.Tensor) else x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        'sigmasq': np.exp(p[0]) / phi2,
        'range_lat': rlon / phi3 ** 0.5,
        'range_lon': rlon,
        'range_time': rlon / phi4 ** 0.5,
        'advec_lat': p[4],
        'advec_lon': p[5],
        'nugget': np.exp(p[6]),
    }


def make_params():
    return [torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True) for v in phys_to_log(INIT_DICT)]


def param_vec_from_params(params):
    return torch.stack([p.detach().reshape(()) for p in params]).to(DEVICE, dtype=DTYPE)


def count_valid(day_map):
    return sum(int((~torch.isnan(v[:, 2])).sum().item()) for v in day_map.values())


def empty_device_cache():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


def round_df(df, digits=ROUND_DECIMALS):
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(digits)
    return out


def save_all(summary_rows, pred_frames, param_rows):
    if summary_rows:
        round_df(pd.DataFrame(summary_rows)).to_csv(OUT_SUMMARY_CSV, index=False, float_format=f'%.{ROUND_DECIMALS}f')
    if pred_frames:
        pred_df = pd.concat(pred_frames, ignore_index=True)
        round_df(pred_df).to_csv(OUT_PRED_CSV, index=False, float_format=f'%.{ROUND_DECIMALS}f')
    if param_rows:
        round_df(pd.DataFrame(param_rows)).to_csv(OUT_PARAM_CSV, index=False, float_format=f'%.{ROUND_DECIMALS}f')


def hybrid_tail_count(model):
    total = 0
    for x in (getattr(model, 'X_A', None), getattr(model, 'X_AB', None), getattr(model, 'X_ABC', None)):
        if x is not None:
            total += int(x.shape[0])
    return total


def st_diag(model):
    batched = getattr(model, 'Batched_Groups', None)
    if batched:
        group_sizes = np.asarray([int(g['target_idx'].shape[0]) for g in batched], dtype=np.int64)
        m_sizes = np.asarray([int(g['max_m']) for g in batched], dtype=np.int64)
        return {
            'n_batches': int(len(batched)),
            'largest_batch_n': int(group_sizes.max()),
            'mean_m': float(m_sizes.mean()),
            'max_m': int(m_sizes.max()),
            'n_tails': int(group_sizes.sum()),
        }
    return {
        'n_batches': np.nan,
        'largest_batch_n': np.nan,
        'mean_m': np.nan,
        'max_m': np.nan,
        'n_tails': hybrid_tail_count(model),
    }

print('Initial log params:', [round(x, 4) for x in phys_to_log(INIT_DICT)])


In [ ]:
# Load full month once.
loader = load_data_dynamic_processed(config.mac_data_load_path)

df_map, ord_mm, nns_map_full, monthly_mean = loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=MM_COND_NUMBER,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=False,
)

sorted_month_keys = sorted(df_map.keys())
day_key_map = {
    day_idx: sorted_month_keys[day_idx * 8:(day_idx + 1) * 8]
    for day_idx in DAY_IDX_LIST
}
selected_keys = [k for day_idx in DAY_IDX_LIST for k in day_key_map[day_idx]]
df_map_selected = {k: df_map[k] for k in selected_keys}

first_key = selected_keys[0]
first_df = df_map_selected[first_key].iloc[ord_mm].reset_index(drop=True)
grid_coords_ordered = first_df[['Latitude', 'Longitude']].values.astype(np.float64)

del df_map
gc.collect()

print(f'Monthly mean TCO: {monthly_mean:.4f}')
print(f'Month time slots loaded then trimmed: {len(sorted_month_keys)} -> {len(selected_keys)}')
print(f'Grid cells: {len(grid_coords_ordered):,}')


In [ ]:
def load_day_map(day_idx, keep_ori=True):
    day_keys = day_key_map[day_idx]
    day_df_map = {k: df_map_selected[k] for k in day_keys}
    hour_indices = [0, len(day_keys)]
    day_map, _ = loader.load_working_data(
        day_df_map, monthly_mean, hour_indices,
        ord_mm=ord_mm, dtype=DTYPE, keep_ori=keep_ori,
    )
    return {k: v.to(DEVICE) for k, v in day_map.items()}, day_keys


def make_holdout(day_map, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    holdout = {}
    train_map = {}
    for h, (key, data) in enumerate(day_map.items()):
        y = data[:, 2].detach().cpu().numpy()
        valid = np.flatnonzero(~np.isnan(y))
        if HOLDOUT_FRAC_PER_HOUR is not None:
            n_hold = int(np.floor(len(valid) * float(HOLDOUT_FRAC_PER_HOUR)))
        else:
            n_hold = int(HOLDOUT_PER_HOUR)
        n_hold = max(1, min(n_hold, len(valid) - 1))
        chosen = np.sort(rng.choice(valid, size=n_hold, replace=False).astype(np.int64))
        train = data.clone()
        train[torch.as_tensor(chosen, device=DEVICE, dtype=torch.long), 2] = torch.nan
        train_map[key] = train
        holdout[key] = chosen
    return train_map, holdout


def flatten_day_map(day_map):
    keys = list(day_map.keys())
    arrays = [day_map[k] for k in keys]
    full = torch.cat(arrays, dim=0).to(DEVICE, dtype=DTYPE).contiguous()
    lengths = [int(a.shape[0]) for a in arrays]
    offsets = np.cumsum([0] + lengths)
    return keys, full, lengths, offsets


def holdout_global_indices(holdout, keys, offsets):
    idx = []
    hour_idx = []
    local_idx = []
    for t, key in enumerate(keys):
        locs = np.asarray(holdout[key], dtype=np.int64)
        idx.extend((offsets[t] + locs).tolist())
        hour_idx.extend([t] * len(locs))
        local_idx.extend(locs.tolist())
    return np.asarray(idx, dtype=np.int64), np.asarray(hour_idx, dtype=np.int64), np.asarray(local_idx, dtype=np.int64)


def fit_variant(day_idx, smooth, spec, train_map):
    date_str = f'{YEAR}{MONTH:02d}{day_idx + 1:02d}'
    n_train = count_valid(train_map)
    print('\n' + '=' * 90)
    print(f"FIT | {spec['variant']} | {date_str} | mean={MEAN_DESIGN} | train valid={n_train:,}")
    print('note:', spec['note'])

    params = make_params()
    if spec['kind'] == 'hybrid':
        model = HybridSTTrendVecchiaFit(
            smooth=smooth,
            input_map=train_map,
            nns_map=nns_map_full,
            mm_cond_number=MM_COND_NUMBER,
            nheads=spec['nheads'],
            limit_A=spec['limit_A'],
            limit_B_local=spec['lag1_local_count'],
            limit_C_local=spec['lag2_local_count'],
            daily_stride=spec['daily_stride'],
            spatial_coords=spec['spatial_coords_for_shift'],
            lag1_lon_offset=spec['lag1_lon_offset'],
            lag1_fresh_count=spec['lag1_fresh_count'],
            lag2_fresh_count=spec['lag2_fresh_count'],
            mean_design=MEAN_DESIGN,
        )
    else:
        model = ColumnSTLagCapTrendVecchiaFit(
            smooth=smooth,
            input_map=train_map,
            mm_cond_number=MM_COND_NUMBER,
            grid_coords=grid_coords_ordered,
            head_right_cols=spec['head_right_cols'],
            above_count=spec['above_count'],
            right_col_count=spec['right_col_count'],
            per_lag_conditioning_counts=spec['per_lag_conditioning_counts'],
            include_lag_self=spec['include_lag_self'],
            target_chunk_size=spec['target_chunk_size'],
            use_data_coords_for_offsets=True,
            mean_design=MEAN_DESIGN,
        )

    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre
    diag = st_diag(model)

    opt = model.set_optimizer(params, lr=LBFGS_LR, max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL, history_size=LBFGS_HIST)
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, opt, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t_fit
    param_vec = param_vec_from_params(params)
    beta = model.get_gls_beta(param_vec).detach()
    est = backmap_params(out)

    fit_info = {
        'date_str': date_str,
        'variant': spec['variant'],
        'model': spec['model'],
        'kernel_kind': spec['kind'],
        'smooth': smooth,
        'mean_design': MEAN_DESIGN,
        'train_loss': float(out[-1]),
        'fit_iter_raw': int(fit_iter),
        'precompute_s': pre_s,
        'fit_s': fit_s,
        'total_fit_s': pre_s + fit_s,
        'n_train': int(n_train),
        'lag_cap_t': spec.get('per_lag_conditioning_counts', (np.nan, np.nan, np.nan))[0],
        'lag_cap_tminus1': spec.get('per_lag_conditioning_counts', (np.nan, np.nan, np.nan))[1],
        'lag_cap_tminus2': spec.get('per_lag_conditioning_counts', (np.nan, np.nan, np.nan))[2],
        **diag,
        **{f'est_{k}': est[k] for k in P_LABELS},
    }
    return model, param_vec, beta, fit_info


In [ ]:
def add_valid(indices, current, valid_np, cap):
    count = 0
    seen = set(current)
    for idx in indices:
        if count >= cap:
            break
        idx = int(idx)
        if idx in seen:
            continue
        if idx < 0 or idx >= len(valid_np):
            continue
        if not valid_np[idx]:
            continue
        current.append(idx)
        seen.add(idx)
        count += 1


def hybrid_conditioning_lists(model, spec, target_globals, hour_idx, local_idx, train_valid_np, lengths, offsets):
    n_points = lengths[0]
    lag1_center = model._build_shift_lookup(n_points, multiplier=1.0)
    lag2_center = model._build_shift_lookup(n_points, multiplier=2.0)
    out = []
    for g, t, local in zip(target_globals, hour_idx, local_idx):
        current = []
        offset = int(offsets[t])
        nbs_current = nns_map_full[local] if local < len(nns_map_full) else np.array([], dtype=np.int64)
        add_valid((offset + nbs_current).tolist(), current, train_valid_np, cap=spec['limit_A'])

        if t > 0:
            prev_off = int(offsets[t - 1])
            prev_len = lengths[t - 1]
            if local < prev_len:
                add_valid([prev_off + local], current, train_valid_np, cap=1)
            local_candidates = [prev_off + int(v) for v in nbs_current if int(v) < prev_len and int(v) != local]
            add_valid(local_candidates, current, train_valid_np, cap=spec['lag1_local_count'])
            center_B = int(lag1_center[local]) if local < len(lag1_center) else int(local)
            if center_B >= prev_len:
                center_B = int(local)
            nbs_B = nns_map_full[center_B] if center_B < len(nns_map_full) else np.array([], dtype=np.int64)
            fresh_B = [prev_off + center_B] + [prev_off + int(v) for v in nbs_B if int(v) < prev_len and int(v) != local]
            add_valid(fresh_B, current, train_valid_np, cap=spec['lag1_fresh_count'])

        if t >= spec['daily_stride']:
            pd_idx = t - spec['daily_stride']
            pd_off = int(offsets[pd_idx])
            pd_len = lengths[pd_idx]
            if local < pd_len:
                add_valid([pd_off + local], current, train_valid_np, cap=1)
            local_candidates = [pd_off + int(v) for v in nbs_current if int(v) < pd_len and int(v) != local]
            add_valid(local_candidates, current, train_valid_np, cap=spec['lag2_local_count'])
            center_C = int(lag2_center[local]) if local < len(lag2_center) else int(local)
            if center_C >= pd_len:
                center_C = int(local)
            nbs_C = nns_map_full[center_C] if center_C < len(nns_map_full) else np.array([], dtype=np.int64)
            fresh_C = [pd_off + center_C] + [pd_off + int(v) for v in nbs_C if int(v) < pd_len and int(v) != local]
            add_valid(fresh_C, current, train_valid_np, cap=spec['lag2_fresh_count'])

        out.append(current)
    return out


def column_conditioning_lists(model, spec, target_globals, hour_idx, local_idx, train_valid_np, lengths, offsets):
    out = []
    lag_caps = tuple(int(x) for x in spec['per_lag_conditioning_counts'])
    for g, t, local in zip(target_globals, hour_idx, local_idx):
        row = int(model._local_to_row[local])
        col = int(model._local_to_col[local])
        spatial_candidates = model._spatial_candidate_locals_uncapped(row, col)
        current = []
        seen = set()
        active_lags = min(len(lag_caps) - 1, int(t)) + 1
        for lag in range(active_lags):
            per_lag_cap = lag_caps[lag]
            if per_lag_cap <= 0:
                continue
            tt = int(t) - lag
            toff = int(offsets[tt])
            added = 0
            if lag > 0 and spec.get('include_lag_self', False):
                gg = toff + int(local)
                if gg not in seen and train_valid_np[gg]:
                    current.append(gg)
                    seen.add(gg)
                    added += 1
            for nb_local in spatial_candidates:
                if added >= per_lag_cap:
                    break
                gg = toff + int(nb_local)
                if gg in seen or not train_valid_np[gg]:
                    continue
                current.append(gg)
                seen.add(gg)
                added += 1
        out.append(current)
    return out


def covariance_from_coords(target_coords, neigh_coords, params, smooth):
    phi1, phi2, phi3, phi4 = torch.exp(params[0:4])
    nugget = torch.exp(params[6])
    advec_lat = params[4]
    advec_lon = params[5]
    sigmasq = phi1 / phi2
    diff = target_coords.unsqueeze(1) - neigh_coords
    d_lat = diff[:, :, 0] - advec_lat * diff[:, :, 2]
    d_lon = diff[:, :, 1] - advec_lon * diff[:, :, 2]
    d_t = diff[:, :, 2]
    dist = torch.sqrt(d_lat.pow(2) * phi3 + d_lon.pow(2) + d_t.pow(2) * phi4 + 1e-8)
    scaled = dist * phi2
    if smooth == 0.5:
        k = sigmasq * torch.exp(-scaled)
    else:
        k = sigmasq * (1.0 + scaled) * torch.exp(-scaled)
    return k, sigmasq, nugget


def cov_nn_from_coords(neigh_coords, params, smooth):
    phi1, phi2, phi3, phi4 = torch.exp(params[0:4])
    nugget = torch.exp(params[6])
    advec_lat = params[4]
    advec_lon = params[5]
    sigmasq = phi1 / phi2
    diff = neigh_coords.unsqueeze(2) - neigh_coords.unsqueeze(1)
    d_lat = diff[:, :, :, 0] - advec_lat * diff[:, :, :, 2]
    d_lon = diff[:, :, :, 1] - advec_lon * diff[:, :, :, 2]
    d_t = diff[:, :, :, 2]
    dist = torch.sqrt(d_lat.pow(2) * phi3 + d_lon.pow(2) + d_t.pow(2) * phi4 + 1e-8)
    scaled = dist * phi2
    if smooth == 0.5:
        K = sigmasq * torch.exp(-scaled)
    else:
        K = sigmasq * (1.0 + scaled) * torch.exp(-scaled)
    m = neigh_coords.shape[1]
    eye = torch.eye(m, device=DEVICE, dtype=DTYPE).unsqueeze(0)
    return K + eye * (nugget + 1e-6)


def predict_holdout(model, spec, params, beta, true_full, train_full, keys, lengths, offsets, holdout):
    target_globals, hour_idx, local_idx = holdout_global_indices(holdout, keys, offsets)
    train_valid_np = (~torch.isnan(train_full[:, 2])).detach().cpu().numpy()
    if spec['kind'] == 'hybrid':
        neigh_lists = hybrid_conditioning_lists(model, spec, target_globals, hour_idx, local_idx, train_valid_np, lengths, offsets)
    else:
        neigh_lists = column_conditioning_lists(model, spec, target_globals, hour_idx, local_idx, train_valid_np, lengths, offsets)

    m_sizes = np.asarray([len(x) for x in neigh_lists], dtype=np.int64)
    keep = m_sizes > 0
    if not np.all(keep):
        print(f"warning: dropping {np.sum(~keep)} heldout targets with no valid conditioning neighbors")
    target_globals = target_globals[keep]
    hour_idx = hour_idx[keep]
    local_idx = local_idx[keep]
    neigh_lists = [x for x, ok in zip(neigh_lists, keep) if ok]
    m_sizes = m_sizes[keep]
    max_m = int(m_sizes.max())

    dummy = torch.zeros((max_m, true_full.shape[1]), device=DEVICE, dtype=DTYPE)
    for k in range(max_m):
        dummy[k, 0] = (k + 1) * 1e8
        dummy[k, 1] = (k + 1) * 1e8
        dummy[k, 3] = (k + 1) * 1e8
    full_pred = torch.cat([train_full, dummy], dim=0)
    dummy_start = int(train_full.shape[0])

    padded = []
    is_dummy_rows = []
    for nbs in neigh_lists:
        pad_n = max_m - len(nbs)
        padded.append([dummy_start + k for k in range(pad_n)] + list(nbs))
        is_dummy_rows.append([True] * pad_n + [False] * len(nbs))
    neigh_idx = torch.as_tensor(np.asarray(padded, dtype=np.int64), device=DEVICE, dtype=torch.long)
    is_dummy = torch.as_tensor(np.asarray(is_dummy_rows, dtype=bool), device=DEVICE)
    target_idx = torch.as_tensor(target_globals, device=DEVICE, dtype=torch.long)

    preds = []
    with torch.no_grad():
        for start in range(0, len(target_globals), PRED_CHUNK_SIZE):
            end = min(start + PRED_CHUNK_SIZE, len(target_globals))
            tidx = target_idx[start:end]
            nidx = neigh_idx[start:end]
            dmask = is_dummy[start:end]
            target_rows = true_full[tidx].to(DTYPE)
            neigh_rows = full_pred[nidx].to(DTYPE)
            coords_t = target_rows[:, [0, 1, 3]].contiguous()
            coords_n = neigh_rows[:, :, [0, 1, 3]].contiguous()
            X_t = model._design_from_rows(target_rows).contiguous()
            X_n = model._design_from_rows(neigh_rows).masked_fill(dmask.unsqueeze(-1), 0.0).contiguous()
            y_n = neigh_rows[:, :, 2].masked_fill(dmask, 0.0).contiguous()

            K = cov_nn_from_coords(coords_n, params, model.smooth)
            k, sigmasq, nugget = covariance_from_coords(coords_t, coords_n, params, model.smooth)
            k = k.masked_fill(dmask, 0.0)
            L = torch.linalg.cholesky(K)
            z = torch.linalg.solve_triangular(L, k.unsqueeze(-1), upper=False)
            w = torch.linalg.solve_triangular(L.transpose(-1, -2), z, upper=True).squeeze(-1)
            resid_n = y_n - torch.einsum('bmf,fp->bm', X_n, beta)
            mean_t = (X_t @ beta).squeeze(-1) + (w * resid_n).sum(dim=1)
            var_t = (sigmasq + nugget) - z.squeeze(-1).pow(2).sum(dim=1)
            var_t = torch.clamp(var_t, min=1e-8)
            y_true = target_rows[:, 2]
            pred_chunk = pd.DataFrame({
                'variant': spec['variant'],
                'model': spec['model'],
                'hour_idx': hour_idx[start:end],
                'local_idx': local_idx[start:end],
                'global_idx': target_globals[start:end],
                'y_true': y_true.detach().cpu().numpy(),
                'y_pred': mean_t.detach().cpu().numpy(),
                'pred_var': var_t.detach().cpu().numpy(),
                'pred_sd': torch.sqrt(var_t).detach().cpu().numpy(),
                'm_used': m_sizes[start:end],
            })
            preds.append(pred_chunk)
    pred = pd.concat(preds, ignore_index=True)
    pred['error'] = pred['y_true'] - pred['y_pred']
    pred['abs_error'] = pred['error'].abs()
    pred['sq_error'] = pred['error'] ** 2
    pred['nlpd'] = 0.5 * (np.log(2 * np.pi * pred['pred_var']) + pred['sq_error'] / pred['pred_var'])
    pred['covered_95'] = pred['abs_error'] <= 1.96 * pred['pred_sd']
    return pred


def summarize_prediction(pred, fit_info):
    rows = []
    groups = [('all', pred)] + [(f'hour_{int(h)+1:02d}', g) for h, g in pred.groupby('hour_idx')]
    for scope, g in groups:
        row = dict(fit_info)
        row.update({
            'eval_scope': scope,
            'n_holdout': int(len(g)),
            'rmse': float(np.sqrt(g['sq_error'].mean())),
            'mae': float(g['abs_error'].mean()),
            'bias': float(g['error'].mean()),
            'nlpd_mean': float(g['nlpd'].mean()),
            'coverage95': float(g['covered_95'].mean()),
            'pred_sd_mean': float(g['pred_sd'].mean()),
            'm_used_mean': float(g['m_used'].mean()),
            'm_used_min': int(g['m_used'].min()),
            'm_used_max': int(g['m_used'].max()),
        })
        rows.append(row)
    return rows


In [ ]:
summary_rows = []
pred_frames = []
param_rows = []

for day_idx in DAY_IDX_LIST:
    true_day_map, day_keys = load_day_map(day_idx, keep_ori=True)
    train_day_map, holdout = make_holdout(true_day_map, seed=RANDOM_SEED + day_idx)
    keys, true_full, lengths, offsets = flatten_day_map(true_day_map)
    _, train_full, _, _ = flatten_day_map(train_day_map)
    n_hold = sum(len(v) for v in holdout.values())
    print('\\n' + '#' * 90)
    print(f'DAY {YEAR}{MONTH:02d}{day_idx+1:02d}: train valid={count_valid(train_day_map):,}, holdout={n_hold:,}')
    print('heldout by hour:', {k: len(v) for k, v in holdout.items()})

    for smooth in FIT_SMOOTHS:
        for variant in RUN_VARIANTS:
            spec = VARIANT_SPECS[variant]
            model, param_vec, beta, fit_info = fit_variant(day_idx, smooth, spec, train_day_map)
            t_pred = time.time()
            pred = predict_holdout(model, spec, param_vec, beta, true_full, train_full, keys, lengths, offsets, holdout)
            pred_s = time.time() - t_pred
            pred['date_str'] = fit_info['date_str']
            pred['smooth'] = smooth
            pred['mean_design'] = MEAN_DESIGN
            pred['prediction_s'] = pred_s
            pred_frames.append(pred)
            rows = summarize_prediction(pred, {**fit_info, 'prediction_s': pred_s})
            summary_rows.extend(rows)
            for p in P_LABELS:
                param_rows.append({
                    'date_str': fit_info['date_str'],
                    'variant': spec['variant'],
                    'model': spec['model'],
                    'parameter': p,
                    'estimate': fit_info[f'est_{p}'],
                })
            save_all(summary_rows, pred_frames, param_rows)
            print('PRED SUMMARY all:', round_df(pd.DataFrame(rows).query("eval_scope == 'all'")))
            del model, param_vec, beta, pred
            empty_device_cache()

summary = pd.DataFrame(summary_rows)
params = pd.DataFrame(param_rows)
preds = pd.concat(pred_frames, ignore_index=True)
print('Saved summary:', OUT_SUMMARY_CSV)
print('Saved predictions:', OUT_PRED_CSV)
print('Saved params:', OUT_PARAM_CSV)
display(round_df(summary.query("eval_scope == 'all'").sort_values(['nlpd_mean', 'rmse'])))


In [ ]:
# Hour-by-hour comparison table.
show_cols = [
    'date_str', 'eval_scope', 'variant', 'n_holdout', 'rmse', 'mae', 'bias',
    'nlpd_mean', 'coverage95', 'pred_sd_mean', 'm_used_mean',
    'est_range_lat', 'est_range_lon', 'est_range_time', 'est_advec_lat', 'est_advec_lon', 'est_nugget',
    'lag_cap_t', 'lag_cap_tminus1', 'lag_cap_tminus2', 'train_loss', 'total_fit_s', 'prediction_s',
]
existing = [c for c in show_cols if c in summary.columns]
display(round_df(summary[existing].sort_values(['eval_scope', 'variant'])))

# Parameter table.
display(round_df(params))
